# ⚛️ Atomic-1Bit — Train Stories Base Model (Colab)

Train the **Stories Base** model (~1.3M params) on [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories).

| Param | Value |
|---|---|
| **Dim** | 256 |
| **Depth** | 6 |
| **Heads** | 4 |
| **Vocab** | 4096 (frequency-filtered) |
| **Context** | 128 |

**Runtime**: Select **GPU** (Runtime → Change runtime type → T4 GPU).

## 1 · Setup

In [ ]:
!pip install -q torch tiktoken datasets numpy matplotlib tqdm pyyaml

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/Atomic-1Bit/weights'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

## 2 · Model Code (Inlined)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

# ---------- BitLinear (1.58-bit) ----------

def activation_quant(x):
    """Quantize activation to INT8 using AbsMax scaling with STE."""
    scale = 127.0 / x.abs().max(dim=-1, keepdim=True)[0].clamp(min=1e-5)
    y = (x * scale).round().clamp(-127, 127)
    y_ste = (y - x * scale).detach() + x * scale
    return y_ste, scale

def weight_quant(w):
    """Quantize weights to {-1, 0, 1} using Mean scaling with STE."""
    scale = 1.0 / w.abs().mean().clamp(min=1e-5)
    y = (w * scale).round().clamp(-1, 1)
    y_ste = (y - w * scale).detach() + w * scale
    return y_ste, scale

class BitLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)
        self.eps = 1e-5

    def forward(self, x):
        x_f32 = x.float()
        rms = torch.sqrt(torch.mean(x_f32 ** 2, dim=-1, keepdim=True) + self.eps)
        x_norm = x_f32 / rms
        x_quant_ste, scale_x = activation_quant(x_norm)
        w_quant_ste, scale_w = weight_quant(self.weight)
        y = F.linear(x_quant_ste, w_quant_ste)
        y_out = y / (scale_x * scale_w)
        if self.bias is not None:
            y_out += self.bias
        return y_out

# ---------- Transformer ----------

@dataclass
class AtomicConfig:
    vocab_size: int = 50257
    dim: int = 512
    depth: int = 8
    heads: int = 8
    context_length: int = 1024

class BitAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.dim % config.heads == 0
        self.dim = config.dim
        self.heads = config.heads
        self.head_dim = config.dim // config.heads
        self.q_proj = BitLinear(config.dim, config.dim)
        self.k_proj = BitLinear(config.dim, config.dim)
        self.v_proj = BitLinear(config.dim, config.dim)
        self.o_proj = BitLinear(config.dim, config.dim)

    def forward(self, x, kv_cache=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.head_dim).transpose(1, 2)
        if kv_cache is not None:
            cached_k, cached_v = kv_cache
            k = torch.cat([cached_k, k], dim=2)
            v = torch.cat([cached_v, v], dim=2)
        new_kv_cache = (k, v)
        T_total = k.shape[2]
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        mask = torch.ones(T, T_total, device=x.device, dtype=torch.bool)
        mask = torch.triu(mask, diagonal=T_total - T + 1)
        att = att.masked_fill(mask, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv_cache

class BitFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = 4 * config.dim
        self.fc1 = BitLinear(config.dim, hidden_dim)
        self.fc2 = BitLinear(hidden_dim, config.dim)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class AtomicBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.RMSNorm(config.dim, eps=1e-5)
        self.attn = BitAttention(config)
        self.ln2 = nn.RMSNorm(config.dim, eps=1e-5)
        self.mlp = BitFeedForward(config)

    def forward(self, x, kv_cache=None):
        attn_out, new_kv_cache = self.attn(self.ln1(x), kv_cache=kv_cache)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x, new_kv_cache

class AtomicTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.dim)
        self.pos_emb = nn.Embedding(config.context_length, config.dim)
        self.layers = nn.ModuleList([AtomicBlock(config) for _ in range(config.depth)])
        self.ln_f = nn.RMSNorm(config.dim, eps=1e-5)
        self.head = BitLinear(config.dim, config.vocab_size)

    def forward(self, idx, kv_cache=None):
        B, T = idx.shape
        if kv_cache is not None and kv_cache[0] is not None:
            pos_offset = kv_cache[0][0].shape[2]
        else:
            pos_offset = 0
        pos = torch.arange(pos_offset, pos_offset + T, dtype=torch.long, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)
        new_kv_cache = []
        for i, layer in enumerate(self.layers):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, new_cache = layer(x, kv_cache=layer_cache)
            new_kv_cache.append(new_cache)
        x = self.ln_f(x)
        logits = self.head(x)
        if kv_cache is not None:
            return logits, new_kv_cache
        return logits

print('✅ Model code loaded.')

## 3 · Dataset

In [ ]:
import json
import numpy as np
import tiktoken
from collections import Counter
from datasets import load_dataset

# -------- Hyperparameters (edit here) --------
BATCH_SIZE   = 32
CONTEXT_LEN  = 128
DIM          = 256
DEPTH        = 6
HEADS        = 4
VOCAB_SIZE   = 4096
UNK_ID       = VOCAB_SIZE - 1
LR           = 1e-3
# ----------------------------------------------

class PocketStoriesDataset:
    def __init__(self, split='train', context_length=128, vocab_file=None):
        if vocab_file is None:
            vocab_file = os.path.join(DRIVE_DIR, 'vocab_map_stories.json')
        print(f'Loading TinyStories ({split})...')
        self.dataset = load_dataset('roneneldan/TinyStories', split=f'{split}[:10%]')
        self.enc = tiktoken.get_encoding('gpt2')
        self.context_length = context_length
        self.vocab_file = vocab_file
        self.token_map = {}
        self.reverse_map = {}
        self._init_vocab()

    def _init_vocab(self):
        if os.path.exists(self.vocab_file):
            print(f'Loading vocab map from {self.vocab_file}...')
            with open(self.vocab_file, 'r') as f:
                data = json.load(f)
                self.token_map = {int(k): v for k, v in data['token_map'].items()}
                self.reverse_map = {int(k): v for k, v in data['reverse_map'].items()}
            print(f'Loaded {len(self.token_map)} mapped tokens.')
            return

        print('Building Frequency-Based Vocab (Scanning first 20k samples)...')
        counter = Counter()
        scan_limit = min(20000, len(self.dataset))
        rows = self.dataset.select(range(scan_limit))
        for text in rows['text']:
            ids = self.enc.encode(text)
            counter.update(ids)
        eot = self.enc.eot_token
        most_common = counter.most_common(VOCAB_SIZE - 2)
        new_id = 0
        valid_gpt_ids = [k for k, v in most_common]
        if eot not in valid_gpt_ids:
            valid_gpt_ids.append(eot)
        valid_gpt_ids = valid_gpt_ids[:VOCAB_SIZE - 1]
        for gpt_id in valid_gpt_ids:
            self.token_map[gpt_id] = new_id
            self.reverse_map[new_id] = gpt_id
            new_id += 1
        self.unk_token = UNK_ID
        print(f'Saving vocab map to {self.vocab_file}...')
        os.makedirs(os.path.dirname(self.vocab_file), exist_ok=True)
        with open(self.vocab_file, 'w') as f:
            json.dump({'token_map': self.token_map, 'reverse_map': self.reverse_map}, f)

    def get_batch(self, batch_size):
        indices = np.random.randint(0, len(self.dataset), batch_size)
        rows = self.dataset.select(indices)
        batch_input_ids, batch_targets = [], []
        for text in rows['text']:
            gpt_ids = self.enc.encode(text)
            gpt_ids.append(self.enc.eot_token)
            pocket_ids = [self.token_map.get(gid, UNK_ID) for gid in gpt_ids]
            if len(pocket_ids) < self.context_length + 1:
                eot_mapped = self.token_map.get(self.enc.eot_token, UNK_ID)
                pocket_ids += [eot_mapped] * (self.context_length + 1 - len(pocket_ids))
            if len(pocket_ids) > self.context_length + 1:
                start = np.random.randint(0, len(pocket_ids) - self.context_length - 1)
                pocket_ids = pocket_ids[start : start + self.context_length + 1]
            batch_input_ids.append(pocket_ids[:-1])
            batch_targets.append(pocket_ids[1:])
        x = torch.tensor(batch_input_ids, dtype=torch.long)
        y = torch.tensor(batch_targets, dtype=torch.long)
        return x, y

print('✅ Dataset class ready.')

## 4 · Training Config

Edit `ADDITIONAL_STEPS` to control how many steps to train.

In [ ]:
# -------- Training length (edit here) --------
ADDITIONAL_STEPS = 5000
USE_AMP = True  # Mixed precision for faster training on GPU
# -----------------------------------------------

## 5 · Train

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

def generate_demo(model, ds, start_text='Once upon a time', max_tokens=60):
    model.eval()
    device = next(model.parameters()).device
    gpt_ids = ds.enc.encode(start_text)
    ids = [ds.token_map.get(gid, UNK_ID) for gid in gpt_ids]
    x = torch.tensor([ids], dtype=torch.long).to(device)
    eot_mapped = ds.token_map.get(ds.enc.eot_token, UNK_ID)
    tokens = []
    for _ in range(max_tokens):
        if x.size(1) >= CONTEXT_LEN:
            break
        with torch.no_grad():
            logits = model(x)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            next_token = torch.multinomial(probs, 1)
            pocket_id = next_token.item()
            gpt_id = ds.reverse_map.get(pocket_id, ds.enc.eot_token)
            try:
                tokens.append(ds.enc.decode([gpt_id]))
            except:
                pass
            x = torch.cat([x, next_token], dim=1)
            if pocket_id == eot_mapped:
                break
    model.train()
    return start_text + ''.join(tokens)

# --- Setup ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

ds = PocketStoriesDataset(context_length=CONTEXT_LEN)
config = AtomicConfig(vocab_size=VOCAB_SIZE, dim=DIM, depth=DEPTH, heads=HEADS, context_length=CONTEXT_LEN)
model = AtomicTransformer(config).to(device)

start_step = 0
ckpt_path = os.path.join(DRIVE_DIR, 'stories_final.pt')
optimizer = optim.AdamW(model.parameters(), lr=LR)

# Resume from checkpoint
if os.path.exists(ckpt_path):
    print(f'>> Resuming from checkpoint: {ckpt_path}')
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
    if 'step' in checkpoint:
        start_step = checkpoint['step']
    if 'optimizer_state_dict' in checkpoint:
        try:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        except:
            print('   Warning: Could not restore optimizer state')
    print(f'   Resuming from step {start_step}')
else:
    print('>> Starting fresh model')

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {total_params:.2f}M')

total_steps = start_step + ADDITIONAL_STEPS
scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and device == 'cuda'))
losses = []

# --- Training Loop ---
print(f'Training from step {start_step} to {total_steps}...')
pbar = tqdm(range(start_step, total_steps), desc='Training')

for step in pbar:
    x, y = ds.get_batch(BATCH_SIZE)
    x, y = x.to(device), y.to(device)

    optimizer.zero_grad()
    with torch.amp.autocast('cuda', enabled=(USE_AMP and device == 'cuda')):
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    loss_val = loss.item()
    losses.append(loss_val)
    pbar.set_postfix(loss=f'{loss_val:.4f}')

    if step % 500 == 0 and step > 0:
        sample = generate_demo(model, ds, 'One day,')
        tqdm.write(f'\n--- Step {step} Sample ---\n{sample}\n')

    if step > 0 and step % 1000 == 0:
        save_dict = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }
        torch.save(save_dict, ckpt_path)
        tqdm.write(f'💾 Checkpoint saved at step {step}')

# Final save
save_dict = {
    'step': total_steps,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}
torch.save(save_dict, ckpt_path)
print(f'\n✅ Training complete! Checkpoint saved to {ckpt_path}')

## 6 · Results

In [ ]:
# --- Loss Curve ---
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, label='Raw')
# Smoothed
window = min(100, len(losses) // 5) if len(losses) > 10 else 1
if window > 1:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(losses)), smoothed, label=f'Smoothed ({window})')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Atomic-1Bit Stories — Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Generate Samples ---
prompts = ['Once upon a time', 'The little dog', 'She was very happy because']
print('\n📝 Generated Samples:')
print('=' * 60)
for p in prompts:
    sample = generate_demo(model, ds, p)
    print(f'\n{sample}')
    print('-' * 60)